# 02 - Exploratory Data Analysis: Drivers & Customers

This notebook explores driver and customer data to identify patterns and potential fraud actors.

## Objectives
- Analyze driver demographics and performance
- Analyze customer demographics and behavior
- Identify high-risk drivers and customers
- Detect potential collusion patterns

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')

from src.etl.extractors import extract_all
from src.etl.transformers import transform_all
from src.features.driver_features import create_driver_features, get_suspicious_drivers
from src.features.customer_features import create_customer_features, get_suspicious_customers
from src.features.aggregations import create_driver_customer_matrix

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Load and transform all data
raw_data = extract_all()
data = transform_all(raw_data)

orders = data['orders']
drivers = data['drivers']
customers = data['customers']

print(f"Orders: {len(orders)} rows")
print(f"Drivers: {len(drivers)} rows")
print(f"Customers: {len(customers)} rows")

## 1. Driver Analysis

In [ ]:
# Create driver features
driver_features = create_driver_features(drivers, orders)
driver_features.head()

In [ ]:
# Driver age distribution
fig = px.histogram(
    driver_features, 
    x='age',
    nbins=20,
    title='Driver Age Distribution',
    labels={'age': 'Age'}
)
fig.show()

In [ ]:
# Driver experience (trips)
fig = px.histogram(
    driver_features, 
    x='trips',
    nbins=30,
    title='Driver Experience (Total Trips)',
    labels={'trips': 'Number of Trips'}
)
fig.show()

In [ ]:
# Missing rate by driver
fig = px.histogram(
    driver_features, 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Driver',
    labels={'missing_rate': 'Missing Rate (%)'}
)
fig.add_vline(
    x=driver_features['missing_rate'].mean(), 
    line_dash='dash', 
    line_color='red',
    annotation_text='Mean'
)
fig.show()

In [ ]:
# Missing rate vs experience
fig = px.scatter(
    driver_features,
    x='trips',
    y='missing_rate',
    color='age_group',
    size='total_orders',
    hover_data=['driver_name', 'total_items_missing'],
    title='Missing Rate vs Experience',
    labels={'trips': 'Total Trips', 'missing_rate': 'Missing Rate (%)'}
)
fig.show()

In [ ]:
# Top suspicious drivers
suspicious_drivers = get_suspicious_drivers(driver_features, missing_rate_threshold=15, min_orders=5)
print(f"Found {len(suspicious_drivers)} suspicious drivers")
suspicious_drivers[['driver_id', 'driver_name', 'age', 'total_orders', 'missing_rate', 'pct_orders_with_missing']].head(10)

In [ ]:
# Missing rate by age group
age_analysis = driver_features.groupby('age_group').agg({
    'missing_rate': 'mean',
    'driver_id': 'count'
}).reset_index()
age_analysis.columns = ['age_group', 'avg_missing_rate', 'driver_count']

fig = px.bar(
    age_analysis,
    x='age_group',
    y='avg_missing_rate',
    title='Average Missing Rate by Driver Age Group',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'age_group': 'Age Group'},
    color='avg_missing_rate',
    color_continuous_scale='Reds'
)
fig.show()

## 2. Customer Analysis

In [ ]:
# Create customer features
customer_features = create_customer_features(customers, orders)
customer_features.head()

In [ ]:
# Customer age distribution
fig = px.histogram(
    customer_features, 
    x='customer_age',
    nbins=20,
    title='Customer Age Distribution',
    labels={'customer_age': 'Age'}
)
fig.show()

In [ ]:
# Missing rate by customer
fig = px.histogram(
    customer_features[customer_features['total_orders'] > 0], 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Customer',
    labels={'missing_rate': 'Missing Rate (%)'}
)
fig.add_vline(
    x=customer_features['missing_rate'].mean(), 
    line_dash='dash', 
    line_color='red',
    annotation_text='Mean'
)
fig.show()

In [ ]:
# Missing rate vs spending
fig = px.scatter(
    customer_features[customer_features['total_orders'] > 0],
    x='total_spent',
    y='missing_rate',
    color='age_group',
    size='total_orders',
    hover_data=['customer_name', 'orders_with_missing'],
    title='Missing Rate vs Total Spending',
    labels={'total_spent': 'Total Spent ($)', 'missing_rate': 'Missing Rate (%)'}
)
fig.show()

In [ ]:
# Top suspicious customers
suspicious_customers = get_suspicious_customers(customer_features, missing_rate_threshold=25, min_orders=2)
print(f"Found {len(suspicious_customers)} suspicious customers")
suspicious_customers[['customer_id', 'customer_name', 'customer_age', 'total_orders', 'missing_rate', 'total_spent']].head(10)

In [ ]:
# Missing rate by customer segment
segment_analysis = customer_features.groupby('customer_segment').agg({
    'missing_rate': 'mean',
    'customer_id': 'count',
    'total_spent': 'sum'
}).reset_index()
segment_analysis.columns = ['segment', 'avg_missing_rate', 'customer_count', 'total_revenue']

fig = px.bar(
    segment_analysis,
    x='segment',
    y='avg_missing_rate',
    title='Average Missing Rate by Customer Segment',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'segment': 'Customer Segment'},
    color='avg_missing_rate',
    color_continuous_scale='Reds'
)
fig.show()

## 3. Driver-Customer Interaction Analysis

In [ ]:
# Create driver-customer interaction matrix
interactions, suspicious_pairs = create_driver_customer_matrix(orders)
print(f"Total unique driver-customer pairs: {len(interactions)}")
print(f"Suspicious pairs: {len(suspicious_pairs)}")

In [ ]:
# Top suspicious pairs (potential collusion)
suspicious_pairs.head(10)

In [ ]:
# Interaction distribution
fig = px.histogram(
    interactions,
    x='interactions',
    nbins=20,
    title='Distribution of Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions'}
)
fig.show()

In [ ]:
# Missing rate in repeated interactions
repeated = interactions[interactions['interactions'] >= 2]

fig = px.scatter(
    repeated,
    x='interactions',
    y='missing_rate',
    size='items_missing',
    hover_data=['driver_id', 'customer_id'],
    title='Missing Rate in Repeated Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions', 'missing_rate': 'Missing Rate (%)'}
)
fig.show()

## 4. Key Findings

### Drivers
- **Total Drivers**: [Fill after running]
- **Suspicious Drivers**: [Fill after running]
- **Highest Risk Age Group**: [Fill after running]

### Customers
- **Total Customers**: [Fill after running]
- **Suspicious Customers**: [Fill after running]
- **Highest Risk Segment**: [Fill after running]

### Collusion Indicators
- **Suspicious Pairs Found**: [Fill after running]
- [Add insights after analysis]